## Miscellanous|

In [4]:
!pip install openpyxl

     ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
     --------- ----------------------------- 61.4/250.9 kB 3.2 MB/s eta 0:00:01
     ----------------- -------------------- 112.6/250.9 kB 1.7 MB/s eta 0:00:01
     ----------------------- -------------- 153.6/250.9 kB 1.3 MB/s eta 0:00:01
     ----------------------------- -------- 194.6/250.9 kB 1.2 MB/s eta 0:00:01
     ----------------------------------- -- 235.5/250.9 kB 1.1 MB/s eta 0:00:01
     -------------------------------------- 250.9/250.9 kB 1.0 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd

labels = pd.read_excel("dataset\\test\\SINEM_NOVO.xlsx")


In [7]:
ground_truth: dict[str, str] = {}

for _, row in labels.iterrows():
    ground_truth[row["New Name"]] = row["Style"]

In [8]:
list(ground_truth.keys()).__len__()

5713

In [11]:
import shutil
import os

start_path: str = "dataset\\test\\Test_set_Cleopatra"
base_end_path: str = "dataset\\test"

for img_name in os.listdir(start_path):
    style_name = ground_truth[img_name]

    actual_end_path: str = os.path.join(base_end_path, style_name)
    os.makedirs(actual_end_path, exist_ok=True)
    
    start_image_path: str = os.path.join(start_path, img_name)
    end_image_path: str = os.path.join(actual_end_path, img_name)
    shutil.move(start_image_path, end_image_path)


In [1]:
import torch.nn.functional as F
from torch import Tensor
import torch

def pairwise_kl(
    logits: Tensor, 
    symmetric: bool = False, 
    reduction: str = "sum", 
    weights: Tensor | None = None
) -> Tensor:
    
    if weights is None:
        weights = torch.ones(
            logits.shape[-1], 
            device=logits.device
        )
    
    red_f = torch.sum if reduction == "sum" else torch.mean
    
    log_prob: Tensor = F.log_softmax(logits, dim=1)       # (B, D)
    prob: Tensor = log_prob.exp()                         # (B, D)

    log_left: Tensor = log_prob.unsqueeze(1)              # (B, 1, D)
    log_right: Tensor = log_prob.unsqueeze(0)             # (1, B, D)

    p_left: Tensor = prob.unsqueeze(1)                    # (B, 1, D)
    p_right: Tensor = prob.unsqueeze(0)                   # (1, B, D)

    kl: Tensor = red_f(
        input=(p_left * (log_left - log_right)) * weights, 
        dim=2
    )  # KL(P_i||P_j)

    if symmetric:
        kl_rev: Tensor = red_f(
            input=(p_right * (log_right - log_left)) * weights, 
            dim=2
        )  # KL(P_j||P_i)
        return kl + kl_rev

    return kl


In [5]:
import torch

inp = torch.randn(4, 4)

inp

tensor([[ 1.1020, -1.6081, -1.2421, -0.8372],
        [-0.3440,  0.8613, -0.2700, -0.0982],
        [ 0.0113,  0.6990,  0.3670,  0.6109],
        [-1.1896, -0.4214,  1.2610,  0.2095]])

In [6]:
print(pairwise_kl(inp))
pairwise_kl(inp, weights=torch.tensor([0.4, 0.5, 0.7, 1]))

tensor([[0.0000, 1.0163, 0.9126, 1.7700],
        [1.1250, 0.0000, 0.0747, 0.6482],
        [0.8791, 0.0732, 0.0000, 0.3650],
        [1.4081, 0.6311, 0.3700, 0.0000]])


tensor([[ 0.0000,  0.3412,  0.2662,  0.6126],
        [ 0.6648,  0.0000, -0.0134,  0.2539],
        [ 0.6590,  0.1135,  0.0000,  0.1638],
        [ 1.0530,  0.4997,  0.2804,  0.0000]])

In [ ]:
pairwise_kl(inp, weights=torch.tensor([0.4, 0.5, 0.7, 1]))

In [29]:
res = pairwise_kl(inp, reduction="mean")

In [26]:
import torch

def kl_similarity(
    logits: Tensor, 
    symmetric: bool = False, 
    reduction: str = "sum"
) -> Tensor:
    kl_div: Tensor = pairwise_kl(
        logits=logits, 
        symmetric=symmetric, 
        reduction=reduction
    )
    return torch.exp(-kl_div)   

In [1]:
from torch import Tensor
import torch

def p_plus_handler(
    target: Tensor, 
    mask_neg_samples: Tensor, 
    logits: Tensor, 
    p_plus: bool
) -> Tensor:
    
    if p_plus:
        pred: Tensor = torch.argmax(
            logits, 
            dim=1
        )

        p_plus_mask: Tensor = (
            pred == target
        )

        p_plus_mask = p_plus_mask.unsqueeze(dim=0).float()

    else: 
        p_plus_mask: Tensor = torch.ones_like(
            target
        ).unsqueeze(dim=0)

        
    mask_pos_samples: Tensor = torch.eq(
        target.unsqueeze(dim=1), 
        target.unsqueeze(dim=0)
    ) * mask_neg_samples

    return mask_pos_samples * p_plus_mask

In [32]:
import torch
from torch import Tensor
from utility import kl_similarity


def forward(
    input: Tensor, 
    target: Tensor,
    weight: Tensor | None = None, 
    p_plus: bool = False
) -> Tensor:
    kl_sim: Tensor = kl_similarity(
        logits=input,
        weight=weight
    )

    mask_neg_samples: Tensor = 1 - torch.eye(
        kl_sim.shape[0], 
        kl_sim.shape[0], 
        dtype=torch.float, 
        device=kl_sim.device
    )

    mask_pos_samples: Tensor = p_plus_handler(
        target=target,
        mask_neg_samples=mask_neg_samples,
        logits=input,
        p_plus=p_plus
    )

    neg_samples: Tensor = (
        kl_sim * mask_neg_samples
    ).sum(dim=-1, keepdim=True)

    pos_samples: Tensor = (
        kl_sim * mask_pos_samples
    )

    log_neg_samples: Tensor = torch.log(neg_samples)
    log_pos_samples: Tensor = pos_samples
    log_pos_samples[pos_samples != .0] = torch.log(pos_samples[pos_samples != .0])

    log_value: Tensor = (
        log_pos_samples 
        - log_neg_samples
    )

    n_sample: Tensor = mask_pos_samples.sum(dim=-1, keepdim=True)
    n_sample[n_sample == .0] += 1

    losses: Tensor = -torch.div(
        input=log_value * mask_pos_samples, 
        other=n_sample
    )

    return losses.sum(dim=[0, 1]) / (mask_pos_samples.sum(dim=[0, 1]) + 1e-6)

In [23]:
input=torch.randn(4, 5)
target=torch.tensor([0, 1, 0, 1])

In [27]:
input

tensor([[-0.1759, -0.0442, -1.1894, -1.2182, -0.3417],
        [-1.3426, -1.0377, -1.5070, -0.1079, -1.7855],
        [ 0.2249,  1.1893,  0.0379,  0.4215, -1.3980],
        [-0.0721, -0.1580, -0.9121, -0.9399,  0.5719]])

In [29]:
print(target)

torch.argmax(
            input, 
            dim=1
        )

tensor([0, 1, 0, 1])


tensor([1, 3, 1, 4])

In [33]:
forward(
    input=input, 
    target=target,
    p_plus=True
)

tensor(0.)

In [ ]:
import torch.nn as nn
from torch import Tensor
import torch

class KL_ContrastiveLoss(nn.Module):
    def __init__(self, symmetric: bool = True, reduction: str = "sum") -> None:
        self.symmetric: bool = symmetric
        self.reduction: str = reduction

    def forward(self, input: Tensor, target: Tensor) -> Tensor:
        kl_sim: Tensor = kl_similarity(
            logits=input,
            symmetric=self.symmetric,
            reduction=self.reduction
        )

        mask_neg_samples: Tensor = 1 - torch.eye(
            kl_sim.shape[0], 
            kl_sim.shape[0], 
            dtype=torch.float, 
            device=kl_sim.device
        )

        mask_pos_samples: Tensor = torch.eq(
            target.unsqueeze(dim=1), 
            target.unsqueeze(dim=0)
        ) * mask_neg_samples

        neg_samples: Tensor = (
            kl_sim * mask_neg_samples
        ).sum(dim=-1, keepdim=True)

        pos_samples: Tensor = (
            kl_sim * mask_pos_samples
        )

        log_neg_samples: Tensor = torch.log(neg_samples)
        log_pos_samples: Tensor = pos_samples
        log_pos_samples[pos_samples != .0] = torch.log(pos_samples[pos_samples != .0])

        log_value: Tensor = (
            log_pos_samples 
            - log_neg_samples
        )

        losses: Tensor = -torch.div(
            input=log_value * mask_pos_samples, 
            other=mask_pos_samples.sum(dim=-1)
        )

        return losses.sum(dim=[0, 1]) / mask_pos_samples.sum(dim=[0, 1])


In [6]:
import torch

a = torch.tensor([0, 1, 0, 5])

res = torch.eq(
    input=a.unsqueeze(dim=0), 
    other=a.unsqueeze(dim=-1)
) # 4 x 4

common_index = torch.nonzero(
    input=res.unsqueeze(dim=-1),
    as_tuple=False
)

In [7]:
res.shape

torch.Size([4, 4])

In [4]:
res[0]

tensor([1, 0, 1, 0], dtype=torch.int32)

In [14]:
logits = torch.tensor([
    [0, 0], 
    [1, 1], 
    [2, 2], 
    [3, 3]
])

logits[res[0]]

tensor([[0, 0],
        [2, 2]])

In [15]:
res

tensor([[ True, False,  True, False],
        [False,  True, False, False],
        [ True, False,  True, False],
        [False, False, False,  True]])

In [17]:
logits.expand(4, 4, 2).shape

torch.Size([4, 4, 2])

In [18]:
logits.expand(4, 4, 2)[res]

tensor([[0, 0],
        [2, 2],
        [1, 1],
        [0, 0],
        [2, 2],
        [3, 3]])

## Graph ensamble

In [1]:
from torch import bmm, Tensor
from typing import NamedTuple
import torch.nn as nn
import torch

class CosineSimBundle(NamedTuple):
    cosine_similarity: Tensor
    avg_cosine_sim: Tensor
    std_cosine_sim: Tensor


def get_cosine_stats(
    patches_emb: Tensor,
    temperature: nn.Parameter,
    valid_patch_mask: Tensor | None = None
) -> CosineSimBundle:
    
    if valid_patch_mask is not None:
        patches_emb = patches_emb * valid_patch_mask.unsqueeze(dim=-1)
        non_valid_cardinality: Tensor = (1 - valid_patch_mask).sum(dim=1)
    else: 
        non_valid_cardinality: float = .0

    patches_emb = patches_emb / (patches_emb.norm(dim=-1, keepdim=True) + 1e-5)
    cosine_similarity: Tensor = bmm(
        input=patches_emb, 
        mat2=patches_emb.transpose(dim0=1, dim1=2)
    )

    cos_identity: Tensor = torch.eye(
        cosine_similarity.shape[1], 
        cosine_similarity.shape[2], 
        device=cosine_similarity.device,
        dtype=cosine_similarity.dtype
    )
    cosine_similarity *= (1 - cos_identity) 
    cosine_similarity /= temperature

    cos_sim_sum: Tensor = cosine_similarity.sum(dim=[1, 2])

    cos_cardinality: Tensor = (cosine_similarity.shape[1])**2 - (cosine_similarity.shape[1] + 2 * non_valid_cardinality) 
    avg_cosine_sim: Tensor = cos_sim_sum / (cos_cardinality + 1e-5)
    avg_cosine_sim = avg_cosine_sim.unsqueeze(dim=-1)

    samples: Tensor = cosine_similarity.flatten(start_dim=1)
    samples_identity: Tensor = cos_identity.flatten(start_dim=0)
    samples_agg: Tensor = (samples - avg_cosine_sim) * (1 - samples_identity)

    var_cardinality: Tensor =  samples_agg.shape[-1]**2 - (samples_agg.shape[-1] + 2 * non_valid_cardinality)
    var_cosine_sim: Tensor = (
        samples_agg**2
    ).sum(dim=1) / (var_cardinality + 1e-5)

    std_cosine_sim: Tensor = var_cosine_sim.sqrt()

    avg_cosine_sim = avg_cosine_sim.view(-1, 1, 1)
    std_cosine_sim = std_cosine_sim.view(-1, 1, 1)

    return CosineSimBundle(
        cosine_similarity=cosine_similarity,
        avg_cosine_sim=avg_cosine_sim,
        std_cosine_sim=std_cosine_sim
    )

In [2]:
patches_emb = torch.tensor([
    [[1., 2., 19.], [3., 4., 19.]],
    [[5., 6., 19.], [7., 8., 19.]]
])

temperature = torch.nn.Parameter(data=torch.tensor(0.07))
valid_mask = torch.tensor([
    [True, False],
    [True, True]
])

In [3]:
sim, mean, std = get_cosine_stats(
    patches_emb,
    temperature, 
    valid_mask.float()
)

In [22]:
from torch import Tensor
import torch


def add_central_nodes_connection(
    edge_mask: Tensor, 
    num_other_expert: int = 2, 
    agg_nodes_id: list[int] | None = None 
) -> Tensor:
    
    suffix_row: Tensor = torch.zeros(
        edge_mask.shape[0], 
        num_other_expert + 1,
        edge_mask.shape[-1],
        device=edge_mask.device,
        dtype=edge_mask.dtype
    )
    suffix_col: Tensor = torch.zeros(
        edge_mask.shape[0], 
        edge_mask.shape[-1] + num_other_expert + 1,
        num_other_expert + 1,
        device=edge_mask.device,
        dtype=edge_mask.dtype
    )

    edge_mask = torch.cat([
            edge_mask,
            suffix_row
        ], 
        dim=1
    )

    edge_mask = torch.cat([
            edge_mask,
            suffix_col
        ], 
        dim=2
    )

    central_node_id: int = edge_mask.shape[-1] - 1

    if agg_nodes_id is None: 
        agg_nodes_id = list(range(central_node_id - num_other_expert, central_node_id))
        agg_nodes_id = [0] + agg_nodes_id

    # Connect the three aggregation nodes 
    # (Vit, masked vit, extrapolated vit) with the central node
    edge_mask[:, central_node_id, agg_nodes_id] = True
    edge_mask[:, agg_nodes_id, central_node_id] = True

    # Connect the three aggregation nodes 
    # with eachother 
    for agg_id in agg_nodes_id: 
        edge_mask[:, agg_id, agg_nodes_id] = True
        edge_mask[:, agg_nodes_id, agg_id] = True

    return edge_mask

In [23]:
from torch_geometric.utils import remove_isolated_nodes
from torch_geometric.data import Batch

def remove_isolated_batch(batch: Batch) -> Batch:
    edge_index, edge_attr, mask = remove_isolated_nodes(
        batch.edge_index,
        num_nodes=batch.num_nodes
    )

    batch.edge_index = edge_index
    if edge_attr is not None:
        batch.edge_attr = edge_attr

    batch.x = batch.x[mask]
    batch.batch = batch.batch[mask]
    return batch


def compute_graph_stats(adjacency: Tensor, valid_patch_mask: Tensor | None) -> tuple[Tensor, Tensor]:
    if valid_patch_mask is not None:
        nodes_cardinality: Tensor = adjacency.shape[1] - (1 - valid_patch_mask).sum(dim=1)
    else:
        nodes_cardinality: Tensor = adjacency.shape[1]

    max_num_edges: Tensor =  0.5 * (nodes_cardinality * (nodes_cardinality - 1))
    num_edges: Tensor = adjacency.sum(dim=[1, 2]) / 2

    return num_edges / max_num_edges, num_edges


In [2]:
import os, sys

# Project root = the folder that contains the "utility/" directory
PROJECT_ROOT = os.path.abspath("..")   # adjust: ".", "..", "../..", etc.
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [24]:
from torch import Tensor

def unify_edge_mask(edge_masks: list[Tensor]) -> Tensor:
    final_edge_mask: list[Tensor] = [edge_masks[0]]

    for idx in range(1, len(edge_masks)):
        final_edge_mask.append(
            torch.cat(
                [torch.ones_like(edge_masks[idk]) for idk in range(idx)]
                +
                [edge_masks[idx]], 
                dim=2
            )
        ) 

        for idj in range(len(final_edge_mask) - 1):
            final_edge_mask[idj] = torch.cat([
                final_edge_mask[idj], 
                torch.ones_like(edge_masks[idj])
            ], dim=2)
           

    return torch.cat(
        final_edge_mask, 
        dim=1
    )


In [8]:
import torch

exp = [
    torch.tensor([[[0, 1], [1, 0]]]), 
    torch.tensor([[[0, 1], [1, 0]]]), 
    torch.tensor([[[0, 1], [1, 0]]]), 
    torch.tensor([[[0, 1], [1, 0]]])
]

In [9]:
res = unify_edge_mask(exp)

In [10]:
exp[0].shape

torch.Size([1, 2, 2])

In [11]:
res.shape

torch.Size([1, 8, 8])

In [26]:
from typing import Literal
import torch.nn as nn
from torch import bmm
from utility.utility import GraphGenout, generate_sub_edge_index, get_raw_edge_mask
from torch_geometric.data import Data

def multiple_generate_connection_discrete(
    patches_emb: list[Tensor], 
    load_param: float,
    temperature: nn.Parameter,
    central_node_mode: Literal["mean", "zero"],
    valid_patch_mask: Tensor | None = None,
    device: str = "cuda", 
    adapt_load_param: bool = False
) -> GraphGenout:
    
    global_nodes: Tensor = torch.cat(patches_emb, dim=1)
    aggregation_nodes: Tensor = global_nodes[:, 0::patches_emb[0].shape[1]]
    
    if central_node_mode == "mean":
        central_node: Tensor = aggregation_nodes.mean(dim=1, keepdims=True)
    else:
        central_node: Tensor = torch.zeros_like(patches_emb[0][:, 0, :])

    global_nodes = torch.cat([global_nodes, central_node], dim=1)

    edge_masks: list[Tensor] = []
    avg_cos_sims: list[Tensor] = []
    std_cos_sims: list[Tensor] = []
    for patch_emb in patches_emb:
        edge_mask, _, avg_cosine_sim, std_cosine_sim = get_raw_edge_mask(
            patches_emb=patch_emb, 
            temperature=temperature,
            valid_patch_mask=valid_patch_mask,
            adapt_load_param=adapt_load_param, 
            load_param=load_param
        )

        edge_masks.append(edge_mask)
        avg_cos_sims.append(avg_cosine_sim.view(-1))
        std_cos_sims.append(std_cosine_sim.view(-1))

    edge_mask: Tensor = unify_edge_mask(edge_masks=edge_masks)

    edge_mask = add_central_nodes_connection(
        edge_mask=edge_mask, 
        num_other_expert=0, 
        agg_nodes_id=list(range(0, global_nodes.shape[1], patches_emb[0].shape[1]))
    )
    
    diagonal_mask = torch.tensor(
        [x for x in range(edge_mask.shape[1])],
        device=device
    )

    adjacency: Tensor = edge_mask.float()
    adjacency[:, diagonal_mask, diagonal_mask] = 0.

    graph_batch: list[Data] = generate_sub_edge_index(
        adjacencies=adjacency,
        x=global_nodes
    )

    grap_density, graph_card = compute_graph_stats(
        adjacency=adjacency, 
        valid_patch_mask=valid_patch_mask.float()
    )

    return GraphGenout(
        graph_batch=graph_batch,
        avg_cosine_sim=torch.stack(avg_cos_sims, dim=1),
        std_cosine_sim=torch.stack(std_cos_sims, dim=1), 
        graph_density=grap_density,
        graph_edges_cardinality=graph_card
    )

In [27]:
multiple_generate_connection_discrete(
    patches_emb=[patches_emb.cpu(), patches_emb.cpu(), patches_emb.cpu()],
    central_node_mode="mean", 
    load_param=0.5,
    temperature=temperature.cpu(),
    valid_patch_mask=valid_mask.cpu(), 
    device="cpu"
)

GraphGenout(graph_batch=[Data(x=[7, 3], edge_index=[2, 30]), Data(x=[7, 3], edge_index=[2, 30])], avg_cosine_sim=tensor([[ 0.0000,  0.0000,  0.0000],
        [14.1820, 14.1820, 14.1820]], grad_fn=<StackBackward0>), std_cosine_sim=tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00],
        [2.8811e-05, 2.8811e-05, 2.8811e-05]], grad_fn=<StackBackward0>), graph_edges_cardinality=tensor([15., 15.]), graph_density=tensor([1.0000, 0.7143]))

In [75]:
from torch import Tensor

def get_least_idx(ensamble_prediction_t: Tensor, most_used_values: Tensor) -> Tensor:

    least_used_map: Tensor = ensamble_prediction_t != most_used_values.unsqueeze(dim=-1)

    # reverse order of each row and then argmax to get the first True
    rev_idx = torch.flip(least_used_map, dims=[1]).int().argmax(dim=1)
    # convert back the reversed index to its real position
    last_idx = least_used_map.size(1) - 1 - rev_idx

    # assign the last learner to the rows with no True
    no_true = ~least_used_map.any(dim=1)
    last_idx[no_true] = ensamble_prediction_t.shape[1] - 1

    return last_idx

In [54]:
import torch
from torch import Tensor
from typing import Literal

def get_basked_representation(
    ensamble_logits: list[Tensor], 
    ensamble_patches: list[Tensor],
    choice: Literal["least", "most", "merge"] = "least"
) -> tuple[Tensor, Tensor]:
    
    ensamble_logits = [en.unsqueeze(dim=1) for en in ensamble_logits]
    ensamble_patches = [en.unsqueeze(dim=1) for en in ensamble_patches]

    ensamble_logits_t: Tensor = torch.cat(ensamble_logits, dim=1)
    ensamble_patches_t: Tensor = torch.cat(ensamble_patches, dim=1)

    ensamble_logits_t = ensamble_logits_t.argmax(dim=-1) # b x n_models

    most_used_values, chosen_idx = torch.mode(ensamble_logits_t, dim=-1)
    
    if choice == "least":
        chosen_idx = get_least_idx(
            ensamble_prediction_t=ensamble_logits_t, 
            most_used_values=most_used_values
        )

    if choice not in ["least", "most"]:
        raise ValueError(f"!!! Mod yet to be developed !!!")

        
    nodes_ids: Tensor = torch.tensor([
        x for x in range(ensamble_logits_t.shape[1])
    ]) 

    chosen_nodes_ids = nodes_ids == chosen_idx.unsqueeze(dim=-1)
    
    final_patches = ensamble_patches_t[chosen_nodes_ids]
    other_global_nodes = ensamble_patches_t[~chosen_nodes_ids]

    other_global_nodes = other_global_nodes.view(
        ensamble_patches_t.shape[0], 
        ensamble_patches_t.shape[1] - 1, 
        ensamble_patches_t.shape[2], -1
    )

    return final_patches, other_global_nodes

In [55]:
ensamble_logits = [
    torch.tensor([
        [0.2, 0.5, 0.8], 
        [0.1, 0.7, 0.36]
        
    ]), 
        torch.tensor([
        [
            0.3, 0.8, 0.7
        ], 
        [
            0.2, 0.4, 0.2
        ]
    ]),
        torch.tensor([
        [
            0.22, 0.53, 0.89
        ], 
        [
            0.13, 0.777, 0.3226
        ]
    ]),

]

ensamble_patches = [
    torch.zeros((2, 16, 29)) for _ in range(3)
]



In [57]:
res1, res2 = get_basked_representation(
    ensamble_logits=ensamble_logits, 
    ensamble_patches=ensamble_patches,
    choice="least"
)

In [58]:
res1.shape

torch.Size([2, 16, 29])

In [59]:
res2.shape

torch.Size([2, 2, 16, 29])

In [66]:
import torch


ten  = torch.tensor(
    [
        [1, 5, 3], 
        [8, 7, 9]
    ]
)

In [67]:
val, idx = torch.mode(ten, dim=-1)

In [68]:
val

tensor([1, 7])

In [69]:
idx

tensor([0, 1])

In [51]:
map = ten != val.unsqueeze(dim=-1)

In [52]:
map.select

<function Tensor.select>

In [70]:
get_least_idx(ten, val)

tensor([2, 2])

In [73]:
torch.flip(ten, dims=[0])

tensor([[8, 7, 9],
        [1, 5, 3]])

In [74]:
ten

tensor([[1, 5, 3],
        [8, 7, 9]])

In [77]:
from torchrl.modules import MLP

mlp = MLP(
            in_features=10,
            out_features=1,
            depth=2
        )

In [ ]:
import torch


x = torch.randn(100, 300)
print(x.shape)
x.requires_grad = True
print(x)

In [ ]:
import torch.nn as nn 

class my_model(nn.Module):
    def __init__(self, in_feature:int, out_feature:int):
        super().__init__()

        self.layer1 = nn.Parameter(torch.randn(in_feature, out_feature) * (1/out_feature))
        self.layer1.requires_grad = True

        self.con_layer = nn.Conv2d

        


In [ ]:
import torch.nn as nn 

class LossFunction(nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    
    def forward(self, x:Tensor, y:Tensor):
        return torch.abs(x-y).mean(dim=0)
        

In [17]:
from torch.utils.data import Dataset
import os
from torchvision import transforms as T
from PIL import Image
from torchvision.io import read_image

class MyDataSet(Dataset):

    def __init__(self, path:str):
        super().__init__()

        self.images_paths = os.listdir(path)
        self.t = T.Compose([
            # T.ToTensor(), 
            T.Resize(224,224)
        ])


    def __len__(self):
        return len(self.images_paths)

    def __getitem__(self, index):
        pth = self.images_paths[index]
        # img = Image.open(pth)
        return self.t(read_image(pth))


    

In [16]:
for name, param in my_model_ins.named_parameters():
    print(name)
    print(param.shape, param.requires_grad)

layer1
torch.Size([12, 120]) True


In [15]:
def get_dataset_cardinality(dataset_path: str, dirs: list[str]) -> dict[str, int]:
    res: dict[str, int] = {}

    for split_name in os.listdir(dataset_path):
        if split_name not in dirs:
            continue

        split_path = os.path.join(dataset_path, split_name)

        for class_name in os.listdir(split_path):
            class_path = os.path.join(split_path, class_name)
            class_cardinality = len(os.listdir(class_path))

            res[class_name] = res.get(class_name, 0) + class_cardinality


    return res



In [26]:
res = get_dataset_cardinality("..\\dataset", ["train"])
res1 = get_dataset_cardinality("..\\dataset", ["valid"])
res2 = get_dataset_cardinality("..\\dataset", ["test"])

In [22]:
res

{'Byzantine': 1736,
 'Cubism': 1807,
 'Egyptian': 1799,
 'Etruscan': 1542,
 'Gothic': 1583,
 'Greek': 1679,
 'Impressionism': 1819,
 'Prehistory': 2081,
 'Renaissance': 1666,
 'Roman': 1631,
 'Surrealism': 1115}

In [23]:
res1

{'Byzantine': 378,
 'Cubism': 362,
 'Egyptian': 441,
 'Etruscan': 327,
 'Gothic': 423,
 'Greek': 447,
 'Impressionism': 397,
 'Prehistory': 291,
 'Renaissance': 412,
 'Roman': 409,
 'Surrealism': 298}

In [24]:
res2

{'Byzantine': 591,
 'Cubism': 549,
 'Egyptian': 532,
 'Etruscan': 463,
 'Gothic': 587,
 'Greek': 574,
 'Impressionism': 549,
 'Prehistory': 429,
 'Renaissance': 538,
 'Roman': 519,
 'Surrealism': 382}

In [27]:
res3 = {
    key: [res[key], res1[key], res2[key], res[key] + res1[key]]
    for key in res
}

res3["Cardinality"] = [sum(res.values()), sum(res1.values()), sum(res2.values()), sum(res.values()) + sum(res1.values())]

In [28]:
import pandas as pd

In [29]:
pd.DataFrame(res3, index=["Train", "Val", "Test", "Train+Val"]).T

,Train,Val,Test,Train+Val
Byzantine,1736,378,591,2114
Cubism,1807,362,549,2169
Egyptian,1799,441,532,2240
Etruscan,1542,327,463,1869
Gothic,1583,423,587,2006
Greek,1679,447,574,2126
Impressionism,1819,397,549,2216
Prehistory,2081,291,429,2372
Renaissance,1666,412,538,2078
Roman,1631,409,519,2040


In [ ]:
pd.DataFrame(res, index=[0]).T

In [8]:
res = {
    "accuracy": [0.596, 0.645, 0.602, 0.45, 0.47], 
    "f1": [0.596, 0.64, 0.6, -1, -1]
}

names = ["base_vit", "extrapolated_vit", "masked_vit", "inception-v4", "elkin"]

In [9]:
import pandas as pd

pd.DataFrame(res, index=names).T

,base_vit,extrapolated_vit,masked_vit,inception-v4,elkin
accuracy,0.596,0.645,0.602,0.45,0.47
f1,0.596,0.640,0.600,-1.00,-1.00
